#Ingest drivers.json file
1. Read the file using spark dataframe reader API
1. Define and enforce schema (preserve the nested structure)
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table  

In [0]:
%run ../00-common/1.Environment-Config

In [0]:
%run ../00-common/2.bronze-helpers

In [0]:
#Define source_file and table_name
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName',StringType()),
])

drivers_schema = StructType([
    StructField('driverId',StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
#read data from contructors file
drivers_df = (
    spark.read
        .format('json')
        .schema(drivers_schema)
        .option('mode','FAILFAST')
        .load(source_file)
)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
(
    drivers_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))